In [ ]:
from cellpose import models
from cellpose.io import imread
import skimage
import os
import scipy
import numpy as np
import matplotlib.pyplot as plt

# MNFinder Test Data

- do it on scaled

In [ ]:
images_dir = "./mnfinder_testing/"
out_dir = images_dir

file_names = [x for x in os.listdir(images_dir) if x.endswith(".phenotype.tif")]
input_files = [f"{images_dir}/{x}" for x in file_names]

imgs = [imread(f) for f in input_files]
nimg = len(imgs)
print(nimg)

model = models.Cellpose(model_type='nuclei')
channels = [0,0]
masks, flows, styles, diams = model.eval(imgs, diameter=None, channels=channels)

output_files = [out_dir + x.replace(".phenotype.tif",".nuclei.tif") for x in file_names]
for i in range(len(masks)):
    skimage.io.imsave(output_files[i], masks[i])

- The ground truth is different in mnfinder data (it's filled)

In [ ]:
images_dir = "./mnfinder_testing/"
out_dir = images_dir

file_names = [x for x in os.listdir(images_dir) if x.endswith(".phenotype.tif")]
micron_files = [f"{images_dir}{x.replace('.tif','_outlines.png')}" for x in file_names]
nuclei_files = [out_dir + x.replace(".phenotype.tif",".nuclei.tif") for x in file_names]

total_overlap = 0
for m, n in zip(micron_files, nuclei_files):
    print(n.split("/")[-1])
    # Load micronuclei and nuclei masks
    mic_masks = skimage.io.imread(m) # here is for mnfinder data
    nuc_masks = skimage.io.imread(n)

    # Prepare micronuclei masks
    mask = np.mean(mic_masks, axis=2) > 0
    edge = skimage.filters.sobel(mask) > 0
    full_mic = mask + edge

    # Dilate micronuclei to obtain extra edges
    d = skimage.morphology.disk(2)
    dilated_obj = skimage.morphology.dilation(full_mic, d)
    dilated_edg = scipy.ndimage.binary_fill_holes(dilated_obj) ^ full_mic

    # Count objects and report differences
    obj = len(np.unique(skimage.morphology.label(mask)))
    full_obj = len(np.unique(skimage.morphology.label(full_mic)))
    dil_obj = len(np.unique(skimage.morphology.label(dilated_obj)))
    print(f"Without and with edges: {obj}(without) - {full_obj}(with) = {obj - full_obj}(diff)")
    print(f"With edges plus dilation: {full_obj}(edges) - {dil_obj}(dilated) = {full_obj - dil_obj}(diff)")

    # Find overlaps and remove them
    overlap = np.logical_and(full_mic, nuc_masks)
    total_overlap += np.sum(overlap)
    print("Overlap:",np.sum(overlap))
    clean_nuclei = (nuc_masks>0) ^ overlap
    clean_labels = skimage.morphology.label(clean_nuclei)
    clean_labels = np.asarray(clean_labels, dtype="uint16")

    # Save result
    skimage.io.imsave(n.replace(".nuclei.tif",".nuclei-clean.tif"), clean_labels)
    # skimage.io.imsave(n.replace(".nuclei.tif",".extra-edge.tif"), dilated_edg)
    

- remeber to remove `nuclei.tif` files

# Test Example

In [ ]:
images_dir = "./mnfinder_testing/"
out_dir = images_dir

file_names = [x for x in os.listdir(images_dir) if x.endswith(".phenotype.tif")]
micron_files = [f"{images_dir}{x.replace('.tif','_outlines.png')}" for x in file_names]
nuclei_files = [out_dir + x.replace(".phenotype.tif",".nuclei.tif") for x in file_names]

# Load micronuclei and nuclei masks
mic_masks = skimage.io.imread(micron_files[0])
nuc_masks = skimage.io.imread(nuclei_files[0])

mask = np.mean(mic_masks, axis=2) > 0
edge = skimage.filters.sobel(mask) > 0
full_mic = mask + edge

plt.imshow(full_mic[300:700,300:700], cmap="gray")

In [ ]:
d = skimage.morphology.disk(2)
dilated_obj = skimage.morphology.dilation(full_mic, d) # dilated micronuclei mask
dilated_edg = scipy.ndimage.binary_fill_holes(dilated_obj) ^ full_mic

edg_mask = np.concatenate([255*edge[:,:,None], 255*dilated_edg[:,:,None], 255*mask[:,:,None]], axis=2)
plt.imshow(edg_mask[300:700,300:700], cmap="gray")

In [ ]:
bright_pixel = np.array([[0, 0, 0, 0, 0],
                         [0, 0, 0, 0, 0],
                         [0, 0, 1, 0, 0],
                         [0, 0, 0, 0, 0],
                         [0, 0, 0, 0, 0]], dtype=np.uint8)
skimage.morphology.dilation(bright_pixel, d)

In [ ]:
overlap = np.logical_and(dilated_obj, nuc_masks) # whether both operands are True (only T & T --> T)
view = np.concatenate([255*overlap[:,:,None], np.zeros_like(mask)[:,:,None], 255*nuc_masks[:,:,None]], axis=2)
plt.imshow(view[300:700,300:700,:], cmap="gray")

In [ ]:
clean_nuclei = (nuc_masks>0) ^ overlap
view = np.concatenate([1.*full_mic[:,:,None], 1*dilated_edg[:,:,None], 1.*clean_nuclei[:,:,None]], axis=2)
plt.imshow(view[300:700,300:700,:])

In [ ]:
x = np.array([
    [False,True,False],
    [True,True,True],
    [False,False,False],
])
y = np.array([
    [False,True,False],
    [True,True,True],
    [False,False,False],
])
np.logical_and(x,y)

# Train Data

In [ ]:
images_dir = "./mnfinder_training/"
out_dir = images_dir

file_names = [x for x in os.listdir(images_dir) if x.endswith(".phenotype.tif")]
micron_files = [f"{images_dir}{x.replace('.tif','_outlines.png')}" for x in file_names]
nuclei_files = [out_dir + x.replace(".phenotype.tif",".nuclei.tif") for x in file_names]

total_overlap = 0
for m, n in zip(micron_files, nuclei_files):
    print(n.split("/")[-1])
    # Load micronuclei and nuclei masks
    mic_masks = skimage.io.imread(m) # here is for mnfinder data
    nuc_masks = skimage.io.imread(n)

    # 4 channels of nuclei mask in mnfinder data
    nuc_masks = nuc_masks[:,:,1]
    
    # Prepare micronuclei masks training data
    mask = np.mean(mic_masks, axis=2) > 0
    edge = skimage.filters.sobel(mask) > 0
    full_mic = mask + edge

    # Dilate micronuclei to obtain extra edges
    d = skimage.morphology.disk(2)
    dilated_obj = skimage.morphology.dilation(full_mic, d)
    dilated_edg = scipy.ndimage.binary_fill_holes(dilated_obj) ^ full_mic

    # Count objects and report differences
    obj = len(np.unique(skimage.morphology.label(mask)))
    full_obj = len(np.unique(skimage.morphology.label(full_mic)))
    dil_obj = len(np.unique(skimage.morphology.label(dilated_obj)))
    print(f"Without and with edges: {obj}(without) - {full_obj}(with) = {obj - full_obj}(diff)")
    print(f"With edges plus dilation: {full_obj}(edges) - {dil_obj}(dilated) = {full_obj - dil_obj}(diff)")

    # Find overlaps and remove them
    overlap = np.logical_and(full_mic, nuc_masks)
    total_overlap += np.sum(overlap)
    print("Overlap:",np.sum(overlap))
    clean_nuclei = (nuc_masks>0) ^ overlap
    clean_labels = skimage.morphology.label(clean_nuclei)
    clean_labels = np.asarray(clean_labels, dtype="uint16")

    # Save result
    skimage.io.imsave(n.replace(".nuclei.tif",".nuclei-clean.tif"), clean_labels)
    # skimage.io.imsave(n.replace(".nuclei.tif",".extra-edge.tif"), dilated_edg)
    

# Train Example

In [ ]:
images_dir = "./mnfinder_training/"
out_dir = images_dir

file_names = [x for x in os.listdir(images_dir) if x.endswith(".phenotype.tif")]
micron_files = [f"{images_dir}{x.replace('.tif','_outlines.png')}" for x in file_names]
nuclei_files = [out_dir + x.replace(".phenotype.tif",".nuclei.tif") for x in file_names]

# Load micronuclei and nuclei masks
mic_masks = skimage.io.imread(micron_files[0])
nuc_masks = skimage.io.imread(nuclei_files[0])

nuc_masks = nuc_masks[:,:,1]
print(mic_masks.shape)
mask = np.mean(mic_masks, axis=2) > 0

# edge = skimage.filters.sobel(mask) > 0
# # edge = mic_masks[:,:,0] > 0 # Use only the red channel
# mask = scipy.ndimage.binary_fill_holes(edge) ^ edge
full_mic = mask

plt.imshow(full_mic[300:700,300:700], cmap="gray")

In [ ]:
d = skimage.morphology.disk(2)
dilated_obj = skimage.morphology.dilation(full_mic, d) # dilated micronuclei mask
dilated_edg = scipy.ndimage.binary_fill_holes(dilated_obj) ^ full_mic

edg_mask = np.concatenate([np.zeros_like(mask)[:,:,None], 255*dilated_edg[:,:,None], 255*mask[:,:,None]], axis=2)
plt.imshow(edg_mask[300:700,300:700,:], cmap="gray")

In [ ]:
overlap = np.logical_and(dilated_obj, nuc_masks) # whether both operands are True (only T & T --> T)
view = np.concatenate([255*overlap[:,:,None], np.zeros_like(mask)[:,:,None], 1*nuc_masks[:,:,None]], axis=2)
plt.imshow(view[300:700,300:700,:], cmap='gray')

In [ ]:
clean_nuclei = (nuc_masks>0) ^ overlap
view = np.concatenate([1.*full_mic[:,:,None], 1*dilated_edg[:,:,None], 1.*clean_nuclei[:,:,None]], axis=2)
plt.imshow(view[300:700,300:700,:])